In [6]:
import pandas as pd

from config import DATA_VERSION_IDENTIFIER, SAVE_FILES, SELECTED_VARIABLES, MUTATION_LABELS_PATH_BM, MUTATION_LABELS_PATH_PB
from core.dummy import save_with_dummy

In [7]:
VENEX_INPUT_FILEPATH = "~/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/clinical_input_data/venex_mutations.csv"
BM_VARIABLE_TABLE_PATH = MUTATION_LABELS_PATH_BM

### **Mutation labels**

In [8]:
df = pd.read_csv(VENEX_INPUT_FILEPATH)

print(f"Number of available samples: {len(df)}")

# get existing mutation variables in VenEx dataset
existing_variables = [variable for variable in SELECTED_VARIABLES if variable in df.columns]

venex_mutation_labels = df[["album_id", "mgg_database"] + existing_variables]
venex_mutation_labels = venex_mutation_labels.rename(columns={"mgg_database": "database_id", "venex_id": "subject_id"})
venex_mutation_labels[existing_variables] = venex_mutation_labels[existing_variables].replace({True: 1, False: 0})


# filter out samples that have been used in model training
df = pd.read_excel(BM_VARIABLE_TABLE_PATH)
venex_mutation_labels = venex_mutation_labels[
    ~venex_mutation_labels.set_index(["album_id", "database_id"]).index.isin(
        df.set_index(["album_id", "database_id"]).index
    )
]
print(f"Found {len(venex_mutation_labels)} BM samples not in model training")

# create dummy subject_id columns
venex_mutation_labels["subject_id"] = venex_mutation_labels.index

display(venex_mutation_labels.head())

#if SAVE_FILES:
#    venex_mutation_labels.to_excel(f"data/{DATE}/venex_mutation_labels_{DATE}.xlsx")

Number of available samples: 95
Found 95 BM samples not in model training


,album_id,database_id,IDH2,NPM1,KIT,TP53,STAG2,subject_id
0,813,vimages_new_06.07.21,0,0,0,0,0,0
1,958,vimages_new_06.07.21,0,0,0,0,0,1
2,1084,vimages_new_06.07.21,0,1,0,0,0,2
3,573,VenEx,1,1,0,0,0,3
4,576,VenEx,0,0,0,0,0,4


### **Anonymized mutation labels**

In [9]:
mutation_map = {
    "RTK_RAS_mutations": "A",
    "NPM1_OR_FLT3_ITD": "B",
    "CpG_mutations": "C",
    "FLT3_mutations": "D",
    "FLT3_ITD": "E",
    "NPM1": "F",
    "MDS_related_mutations": "G",
    "Transcription_mutations": "H",
    "DNMT3A": "I",
    "chromatin_cohesin_mutations": "J",
    "IDH_mutations": "K",
    "complex_karyotype": "L",
    "TET2": "M",
    "Spliceosome_mutations": "N",
    "monosomal_karyotype": "O",
    "RAS_mutations": "P",
    "IDH2": "Q",
    "trisomy_add_dup_amp_8": "R",
    "monosomy_del_7": "S",
    "TP53_OR_monosomy17_del_17p": "T",
    "monosomy5_del_5q": "U",
    "RUNX1": "V",
    "NRAS": "W",
    "SRSF2": "X",
    "TP53": "Y",
    "BCOR_mutations": "Z",
    "FLT3_TKD": "AA",
    "BCOR": "AB",
    "ASXL1": "AC",
    "RUNX1-RUNX1T1_OR_t_8_21": "AD",
    "monosomy17_del_17p": "AE",
    "WT1": "AF",
    "IDH1": "AG",
    "CEBPA": "AH",
    "PTPN11": "AI",
    "translocation11": "AJ",
    "KRAS": "AK",
    "JAK2": "AL",
    "GATA_mutations": "AM",
    "STAG2": "AN",
    "translocation3": "AO",
    "GATA2": "AP",
    "NF1": "AQ",
    "CBL": "AR",
    "PHF6": "AS",
    "U2AF1": "AT",
    "KIT": "AU",
    "KMT2A_rearrangement": "AV",
    "CBFB-MYH11": "AW",
    "EP300": "AX",
    "EZH2": "AY",
    "PML-RARA": "AZ"
}

In [10]:
venex_mutation_labels_anonymized =  venex_mutation_labels.copy()
for variable in existing_variables:
    venex_mutation_labels_anonymized = venex_mutation_labels_anonymized.rename(columns={variable: mutation_map[variable]})

display(venex_mutation_labels_anonymized.head())

if SAVE_FILES:
    V = DATA_VERSION_IDENTIFIER
    save_with_dummy(venex_mutation_labels_anonymized, f"data/{V}/mutation_labels/venex_bm_samples_{V}.xlsx", id_columns=["subject_id", "album_id", "database_id"])

,album_id,database_id,Q,F,AU,Y,AN,subject_id
0,813,vimages_new_06.07.21,0,0,0,0,0,0
1,958,vimages_new_06.07.21,0,0,0,0,0,1
2,1084,vimages_new_06.07.21,0,1,0,0,0,2
3,573,VenEx,1,1,0,0,0,3
4,576,VenEx,0,0,0,0,0,4


Saved: data/03-06-2026/mutation_labels/venex_bm_samples_03-06-2026.xlsx
Saved: data/03-06-2026/mutation_labels/venex_bm_samples_03-06-2026_DUMMY.xlsx


In [11]:
df = pd.read_excel(MUTATION_LABELS_PATH_PB)
df.head()

,subject_id,album_id,database_id,A,B,C,D,E,F,G,...,AQ,AR,AS,AT,AU,AV,AW,AX,AY,AZ
0,DUMMY_0000,DUMMY_0000,DUMMY_0000,NaN,NaN,1.0,1.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,1.0
1,DUMMY_0001,DUMMY_0001,DUMMY_0001,1.0,0.0,0.0,NaN,0.0,1.0,NaN,...,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.0
2,DUMMY_0002,DUMMY_0002,DUMMY_0002,1.0,1.0,0.0,0.0,NaN,1.0,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,DUMMY_0003,DUMMY_0003,DUMMY_0003,0.0,1.0,NaN,1.0,0.0,1.0,NaN,...,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0
4,DUMMY_0004,DUMMY_0004,DUMMY_0004,NaN,NaN,0.0,NaN,1.0,1.0,0.0,...,1.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,NaN
